# Giai đoạn 3 — Dung hợp Dữ liệu (Combined)

**Mục tiêu:** Gộp cả 5 tập augmented độc lập (`aug_baseline.csv`, `aug_eda.csv`, `aug_llm.csv`, `aug_bt.csv`, `aug_voz.csv`) thành 1 tập train "trùm cuối" cho thí nghiệm `phobert_combined`.

**Input:** `data/processed/dev.csv`, `test.csv` (màng lọc leakage) + 5 file `data/augmented/aug_*.csv`

**Output:** `data/augmented/aug_combined.csv`

Có 3 bước lọc bắt buộc, theo đúng thứ tự:
1. **Loại nhãn xung đột** — cùng câu nhưng khác nhãn giữa các nguồn augmentation khác nhau (chạy trước dedupe, vì không thể tin nhãn của câu này dù giữ bản nào).
2. **Loại trùng lặp chéo** giữa các phương pháp (vd. baseline đã có sẵn trong cả `aug_eda.csv` lẫn `aug_llm.csv`).
3. **Loại leakage** với `dev.csv`/`test.csv` — bắt buộc vì `aug_bt.csv`/`aug_voz.csv` sinh dữ liệu mới, không có gì đảm bảo không vô tình trùng.

Cả 3 bước dùng `text_key()` chuẩn hoá (NFKC + lowercase + gộp khoảng trắng) — không so `text_raw` nguyên văn, tránh bỏ sót các câu chỉ khác hoa/thường hoặc khoảng trắng thừa.

## 1. Dependencies & Cấu hình

In [ ]:
import re
import unicodedata
from pathlib import Path
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
AUGMENTED_DIR = DATA_DIR / "augmented"

AUG_FILES = ["aug_baseline.csv", "aug_eda.csv", "aug_llm.csv", "aug_bt.csv", "aug_voz.csv"]

def text_key(text):
    """Chuẩn hoá NFKC + lowercase + gộp khoảng trắng trước khi so trùng/leakage --
    dùng đúng 1 hàm này xuyên suốt dự án, không so text_raw nguyên văn."""
    text = unicodedata.normalize("NFKC", str(text)).strip().lower()
    return re.sub(r"\s+", " ", text)

Mounted at /content/drive


## 2. Tạo Màng lọc Leakage từ Dev/Test

`dev.csv`/`test.csv` là bộ "đóng băng" -- chỉ đọc, không sửa. Nếu file nào thiếu cột `text_raw` (trường hợp cũ), tái tạo tạm bằng cách bỏ dấu `_` khỏi `text`.

In [ ]:
df_dev = pd.read_csv(PROCESSED_DIR / "dev.csv")
df_test = pd.read_csv(PROCESSED_DIR / "test.csv")

def get_safe_text_raw(df):
    if "text_raw" not in df.columns:
        raw = df["text"].astype(str).str.replace("_", " ")
        return raw.str.replace(r"\s+", " ", regex=True).str.strip()
    return df["text_raw"].astype(str)

protected_keys = (
    {text_key(t) for t in get_safe_text_raw(df_dev)}
    | {text_key(t) for t in get_safe_text_raw(df_test)}
)
print(f"Số key được bảo vệ (dev + test): {len(protected_keys):,}")

Số key được bảo vệ (dev + test): 9,131


## 3. Load & Làm sạch chuẩn 4 cột

Xoá cột rác, tái tạo `text_raw` nếu thiếu, loại `NaN`, ép kiểu chuỗi. Không dùng `rglob` tìm file tự động -- đọc trực tiếp đúng đường dẫn để tránh chọn nhầm bản cũ nếu trùng tên file ở nhiều thư mục con.

In [ ]:
def load_and_clean(filepath):
    df = pd.read_csv(filepath)

    junk_cols = [c for c in df.columns if c.startswith("__") or c.startswith("Unnamed")]
    if junk_cols:
        df = df.drop(columns=junk_cols)

    if "text_raw" not in df.columns:
        df["text_raw"] = df["text"].astype(str).str.replace("_", " ")
        df["text_raw"] = df["text_raw"].str.replace(r"\s+", " ", regex=True).str.strip()

    df = df.dropna(subset=["text", "text_raw", "label"]).reset_index(drop=True)
    df["text"] = df["text"].astype(str)
    df["text_raw"] = df["text_raw"].astype(str)

    if "source" not in df.columns:
        df["source"] = filepath.stem.replace("aug_", "")

    df["_key"] = df["text_raw"].map(text_key)
    return df[["text", "text_raw", "label", "source", "_key"]]

df_list = []
for filename in AUG_FILES:
    filepath = AUGMENTED_DIR / filename
    if filepath.exists():
        cleaned = load_and_clean(filepath)
        df_list.append(cleaned)
        print(f"Đã nạp: {filename} ({len(cleaned):,} dòng)")
    else:
        print(f"Không tìm thấy {filepath} -- kiểm tra lại đường dẫn hoặc tên file.")

df_combined = pd.concat(df_list, ignore_index=True)
len_total = len(df_combined)
print(f"\nTổng số dòng trước khi lọc: {len_total:,}")

Đã nạp: aug_baseline.csv (85,127 dòng)
Đã nạp: aug_eda.csv (134,412 dòng)
Đã nạp: aug_llm.csv (88,191 dòng)
Đã nạp: aug_bt.csv (4,234 dòng)
Đã nạp: aug_voz.csv (803 dòng)

Tổng số dòng trước khi lọc: 312,767


## 4. Loại Nhãn xung độtCùng 1 câu (`_key` giống nhau) nhưng khác `label` giữa các nguồn -- không tin được nhãn của những câu này, loại khỏi TOÀN BỘ (không giữ bản nào).

In [ ]:
conflict_keys = set(
    df_combined.groupby("_key")["label"].nunique().loc[lambda c: c > 1].index
)
if conflict_keys:
    print(f"Phát hiện {len(conflict_keys):,} câu có nhãn xung đột giữa các nguồn -- loại bỏ toàn bộ.")
    df_combined = df_combined[~df_combined["_key"].isin(conflict_keys)].reset_index(drop=True)
else:
    print("Không phát hiện nhãn xung đột.")

len_after_conflict = len(df_combined)

Phát hiện 55 câu có nhãn xung đột giữa các nguồn -- loại bỏ toàn bộ.


## 5. Loại Trùng lặp chéo & LeakageDùng `_key` đã chuẩn hoá cho cả 2 bước -- không so `text_raw` nguyên văn.

In [ ]:
df_combined = df_combined.drop_duplicates(subset=["_key"]).reset_index(drop=True)
len_after_dedup = len(df_combined)

is_leakage = df_combined["_key"].isin(protected_keys)
df_combined = df_combined[~is_leakage].drop(columns=["_key"]).reset_index(drop=True)
len_final = len(df_combined)

## 6. Báo cáo & Lưu kết quả

In [ ]:
print("-" * 40)
print("BÁO CÁO GỘP DỮ LIỆU:")
print(f"Tổng số dòng ban đầu        : {len_total:,}")
print(f"Bị loại do nhãn xung đột    : {len_total - len_after_conflict:,}")
print(f"Bị loại do trùng lặp nội bộ : {len_after_conflict - len_after_dedup:,}")
print(f"Bị loại do rò rỉ (Leakage)  : {len_after_dedup - len_final:,}")
print(f"TỔNG SỐ DÒNG HỢP LỆ CUỐI    : {len_final:,}")

print("\nPhân bố nhãn cuối cùng:")
print(df_combined["label"].value_counts())

print("\nPhân bố theo nguồn:")
print(df_combined["source"].value_counts())

out_path = AUGMENTED_DIR / "aug_combined.csv"
df_combined.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\nĐã lưu tập gộp hoàn chỉnh vào: {out_path}")

----------------------------------------
BÁO CÁO GỘP DỮ LIỆU:
Tổng số dòng ban đầu        : 312,767
Bị loại do nhãn xung đột    : 230
Bị loại do trùng lặp nội bộ : 170,304
Bị loại do rò rỉ (Leakage)  : 677
TỔNG SỐ DÒNG HỢP LỆ CUỐI    : 141,556

Phân bố nhãn cuối cùng:
label
OFFENSIVE    62118
CLEAN        59305
HATE         20133
Name: count, dtype: int64

Phân bố theo nguồn:
source
baseline    84300
eda         49175
bt           4232
llm          3064
voz           752
youtube        33
Name: count, dtype: int64

Đã lưu tập gộp hoàn chỉnh vào: /content/drive/MyDrive/Hate_Speech_Detection/data/augmented/aug_combined.csv
